# Лабораторная работа 8. 
## Кластеризация методом k-means

Сегодня мы поработаем датасетом о винах из пакете sklearn. Данный датасет обычно используют для классификации, но мы не будем загружать данные об истинных метках класса (о производителях вина), а с помощью методов кластеризации посмотрим, образуют ли эти вина какие-то группы по своему химическому составу. 

Для анализа будем использовать 2 наиболее популярных метода кластеризации: метод k средних и метод иерархической агломеративной кластеризации.

**Обзор доступных данных**

В выборке 178 наблюдений и 13 переменных. Целевой переменной является производитель вина. 



| Имя столбца          | Значение                             |
|:-----------------   :|:------------------------------------:|
| alcohol              | Алкоголь                             |
| malic_acid           | Яблочная кислота                     |
| ash                  | Пепел                                |
| alcalinity_of_ash    | Щелочность золы                      |
| magnesium            | Магний                               |
| total_phenols        | Общие фенолы                         |
| flavanoids           | Флаваноиды                           |
| nonflavanoid_phenols | Нефлаваноидные фенолы                |
| proanthocyanins      | Проантоцианы                         |
| color_intensity      | Интенсивность цвета                  |
| hue                  | Оттенок                              |
| od280/od315          | OD280 / OD315 разбавленных вин       |
| proline              | Пролин                               |

### Загрузим библиотеки

In [1]:
from sklearn.datasets import load_wine #датасет содержится в библиотеке scikit-learn
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.preprocessing import StandardScaler #метод z-нормализации переменных 
from sklearn.cluster import KMeans # реализация метода k средних в sklearn
from sklearn.metrics import silhouette_score, classification_report

### Загрузим данные

In [4]:
data = load_wine()
df = pd.DataFrame(data.data, columns=data.feature_names) # в датасет df запишем только значения предикторов. Значения откликов мы отбрасываем
df

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0


### 📌📌📌 Задание 1. Предварительный анализ данных

1. Выясните, есть ли в датасете  пропуски и какого типа предикторы
2. Построив для каждой переменной диаграммы размаха (Boxplots), выясните, имеются ли выбросы по значениям переменных

In [ ]:
## Ваш код

### Проведем стандартизацию
Переменные датасета могут иметь разный масштаб (например, в нашем датасете переменная `magnesium` принимает значения  на \[70,162\], а переменная `malic_acid` - на  \[0.74-5.8\]). 

Так как метод k-means в качестве меры близости использует функцию расстояния, то переменные с большим масштабом будут вносить больший вклад в результаты кластеризации. Если переменные одинаково важны для решения задачи, то небходимо привести их к одному масштабу.

Существует несколько способов стандартиации (нормализации переменных). Изучите информацию о них по ссылкам:
- [z-стандартизация](https://www.helenkapatsa.ru/standartizatsiia/)
- [минимакс-стандартизация](https://www.helenkapatsa.ru/normalizatsiia/)

В нашей задаче будем использовать z-стандартизацию, которая реализована  в пакете `sklearn.preprocessing` с помощью функции `StandardScaler()`

In [ ]:
scaler = StandardScaler()
dff = scaler.fit_transform(df) # обучение и одновременное применение стандартизатора к данным df
dff = pd.DataFrame(dff, columns=df.columns)# датасет из стандартизованных переменных
dff.head()

### Определим, имеется ли в данных тенденция к кластеризации (с помощью статистики Хопкинса)
В Python на данный момент нет встроенного метода вычисления статистики Хопкинса, поэтому напишем пользовательную функцию.

In [26]:
# X - выборка, для которой вычисляется статистика Хопкинса
# knn - число  ближайших соседей для вычисления статистики
def hopkins(X,knn):
    
    from sklearn.neighbors import NearestNeighbors
    from random import sample
    from numpy.random import uniform
    import numpy as np
    from math import isnan
    
    d = X.shape[1]
    #d = len(vars) # columns
    n = len(X) # rows
    m = int(0.1 * n) 
    nbrs = NearestNeighbors(n_neighbors=knn).fit(X.values)
 
    rand_X = sample(range(0, n, 1), m)# генерируется псевдонабор данных, имеющих равномерное распределение
 
    ujd = []
    wjd = []
    for j in range(0, m):
        u_dist, _ = nbrs.kneighbors(uniform(np.amin(X,axis=0),np.amax(X,axis=0),d).reshape(1, -1), 2, return_distance=True)
        ujd.append(u_dist[0][1])
        w_dist, _ = nbrs.kneighbors(X.iloc[rand_X[j]].values.reshape(1, -1), 2, return_distance=True)
        wjd.append(w_dist[0][1])
 
    H = sum(ujd) / (sum(ujd) + sum(wjd))
    if isnan(H):
        print(ujd, wjd)
        H = 0
 
    return H

### 📌📌📌 Задание 2. Вычисление статистики Хопкинса и анализ результатов
**Варианты 1-3**:

Вычислите статистику Хопкинса для датасета `dff`, взяв в число ближайших соседей, равным:
- **Вариант 1**: 3
- **Вариант 2**: 4
- **Вариант 3**: 5

Сделайте вывод о наличии/отсутствии тенденции к кластеризации.

**Варианты 4**:
Как видно из синтаксиса функции `hopkins(X,knn)`, значение статистики будет зависить от псевдонабора данных, который генерируется в процессе вычисления статистики. 

Будет правильнее вычислить статитику Хопкинса несколько раз (т.е. для нескольких случайных псевдонаборов), а потом принять решение о тенденции о кластеризации, взяв среднее значение статистики. 

Реализуйте это. Самостоятельно примите решение о количестве вычислений статистике и числе ближайших соседей.


In [2]:
### Ваш код

## Определение наилучшего числа кластеров методом локтя

В методе k-means одним из внешних параметров является число кластеров. Правильный выбор числа кластеров - залог качественной кластеризации. Число кластеров можно выбирать как исходя из постановки задачи, так с помощью методов, оценивающих кластеризуемость выборки.

В нашей работе будем использовать метод локтя. Для того, чтобы его реализовать, нам будет необходимо провести несколько кластеризаций методом k-means (выбирая число кластеров из некоторого диапазона).

Функция `KMeans(n_clusters=i,...)` пакета `sklearn.cluster` одновременно вычисляет **общий внутникластерный разброс** (`Kmeans.inertia_`), который и является функцией, визуализируемой в методе локтя.

In [ ]:
inertia = []
num_of_clusters = np.arange(1,14) # наилучшее число клатеров ищем в диапазоне от 1 до 14
for i in num_of_clusters:
    km = KMeans(n_clusters=i, max_iter=100, random_state=100)
    km.fit(dff)    
    inertia.append(km.inertia_)
    
plt.plot(num_of_clusters, inertia)
plt.title('Elbow plot')
plt.xlabel('Количество кластеров')
plt.ylabel('Внутриклассовый разброс')
plt.show()

**Вопрос:** Какое количество кластеров можно считать оптимальным?

### 📌📌📌 Задание 3. Кластеризация с помощью KMeans
При выполнении задания пользуйтесь справкой по методу k-means: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

После того, как вы определили оптимальное число кластеров, можно приступать к кластеризации. 

**Для всех вариантов:**
1. Проведите кластеризацию методом k-means и запишите ее результаты в отдельный столбец датасета `df`.
2. Выведите на экран информацию о количестве объектов в каждом кластере.
3. Постройте диаграммы размаха значение указанных в вашем варианте переменных по каждому из полученных кластеров:

    **Вариант 1**: Алкоголь и яблочная кислота

    **Вариант 2**: Магний и общие фенолы

    **Вариант 3**: Флавоноиды и оттенок

    **Вариант 4**: Проантоцианы и пролин

    Оба ли признака являются значимыми для кластеризации?

4. **Только для варианта 4**: вина представленные в датасете изготовлены тремя различными производителями (номера производителей - это метки классов в `data.target`)
Если разбить вина на 3 кластера, то не получится ли при этом, что мы кластеризуем по производителям? Исследуйте, насколько кластеры будут совпадать с классами. При этом нужно иметь в виду, что нумерации кластеров и классов могуть не совпадать, поэтому испробуйте все возможные способы перенумеровать кластеры.

Проинтерпретируйте полученные результаты.

In [3]:
## Ваш код